In [ ]:
import json
from IPython.display import display, Markdown, Latex
import pycountry_convert as pc
import yaml as yml
from matplotlib_inline.backend_inline import set_matplotlib_formats
import pandas as pd

from emu_renewal.constants import ANALYSIS_NAMES, DATA_PATH, FULL_RUN
from emu_renewal.outputs import get_param_vals_by_analysis, get_prop_better
from emu_renewal.plotting import plot_kde_comparison
from emu_renewal.utils import get_country_name, ANALYSIS_TYPES, get_countries_by_continent, split_list_into_segments, get_analysis_paths, get_analysis_commits_df

set_matplotlib_formats("svg")

In [ ]:
all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json", "r"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
avail_countries = [iso3 for iso3, v in analysis_paths.items() if "oxcgrt_floored" in v]
countries_by_cont = get_countries_by_continent(avail_countries)
disp_posts = {}
req_analyses = ANALYSIS_TYPES
for iso3 in avail_countries:
    c_paths = {k: v for k, v in analysis_paths[iso3].items() if k in req_analyses}
    disp_posts[iso3] = get_param_vals_by_analysis("dispersion_proc", c_paths)

# Purpose
This document presents results based on the posterior distribution 
of the residual transmission scaling dispersion parameter.
This quantity governs the distribution in the change in residual scaling for transmission
from one value in the process series to the subsequent update (in log space).
As such, smaller values imply that smaller updates could still
result in good calibrations.
Lower values for this parameter can therefore represent 
less dramatic changes needing to be applied through 
the non-mechanistic component of the model. 
As such, we interpret mobility analysis approaches 
for which this posterior distribution of this parameter
was lower as being a more plausible representation of reality.
This is intended as a more formal quantification of the 
differences in variation in the non-mechanistic transmission scaling
presented in the previous document.

# Dispersion parameter distributions by analysis and country

# Proportion better than baseline

For each country and scaled analysis, randomly shuffle the baseline
(`no_scaling`) posterior draws and ask what proportion of paired runs
have a lower dispersion under the scaled analysis compared to the baseline.
Values above 0.5 favour the scaled analysis, values below favour baseline.


In [ ]:

ordered_countries = [iso3 for iso3s in countries_by_cont.values() for iso3 in iso3s]
rows = {}
for iso3 in ordered_countries:
    country = get_country_name(iso3)
    c_disps = disp_posts[iso3]
    analyses = [a for a in c_disps.columns if a != "no_scaling"]
    rows[country] = {a: get_prop_better(c_disps, a, "no_scaling") for a in analyses}

prop_better_table = pd.DataFrame.from_dict(rows, orient="index")
prop_better_table = prop_better_table.map("{:.3f}".format).replace("nan", "no analysis")
prop_better_table = prop_better_table.rename(columns=ANALYSIS_NAMES)
display(Markdown(prop_better_table.to_markdown(disable_numparse=True)))


# Comparison of posterior distributions
## Policy analyses

In [ ]:
param = "dispersion_proc"
param_name = yml.safe_load(open(DATA_PATH / "evidence/priors.yml", "r"))["other"]["dispersion_proc"]["short_name"]

oxcgrt_disp_posts = {iso3: disp_posts[iso3][["no_scaling", "oxcgrt_floored", "oxcgrt_independent"]] for iso3, v in disp_posts.items()}
for c, (cont, cont_countries) in enumerate(countries_by_cont.items()):
    if c:
        display(Latex(r"\newpage"))
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"## {cont_name}"))
    title = f"Posterior distribution for {param_name} parameter, {cont_name}"
    for countries in split_list_into_segments(cont_countries, 16):
        param_posts = {iso3: oxcgrt_disp_posts[iso3] for iso3 in countries}
        display(plot_kde_comparison(param_posts)) 

## All analysis types

In [ ]:
for c, (cont, cont_countries) in enumerate(countries_by_cont.items()):
    if c:
        display(Latex(r"\newpage"))
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"## {cont_name}"))
    title = f"Posterior distribution for {param_name} parameter, {cont_name}"
    for countries in split_list_into_segments(cont_countries, 16):
        param_posts = {iso3: disp_posts[iso3] for iso3 in countries}
        display(plot_kde_comparison(param_posts)) 

{{< pagebreak >}}

# Commits used for analyses
For reproducibility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())